In [1]:
#### mv packages ####
import modules.data as d
import modules.model as m
import modules.pooling as p
import modules.train as t
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device, generator = u.Devices().auto_set_device()#['cuda:1', 'cuda:0'])
# device, generator = u.Devices().set_device('cpu')

#### data ####
brca = d.Preprocessor(
    tcga_project='TCGA-BRCA',
    tcga_dir=dataset_dir/'tcga',
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',
    
    # counts
    apply_DESeq_norm=False, 
    log_transform=False,
    scale_method=None,

    # etc
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Primary Tumor']},
    max_subset = 120,
)
_dataset = d.GraphDataset(brca)
_batch = d.get_toy_databatch(_dataset, generator)

# #### Device() ####
# device = cuda:4

# #### Preprocessor() ####
# log0_method              log1p                    str
# class_weights            (6,)                     Tensor (cuda:4)
# edge_index               (2, 32798)               Tensor (cuda:4)
# edge_attr                (32798, 16)              Tensor (cuda:4)
# gene_counts              (4383, 562)              DataFrame
# metadata                 (562, 3)                 DataFrame
# relation                 (32798, 18)              DataFrame
# node_id_map              4383                     dict
# mask_list                305                      list
# mask                     (4383, 305)              Tensor (cuda:4)
# x                        (562, 4383, 1)           Tensor (cuda:4)
# y                        (562,)                   Tensor (cuda:4)
# y_labels                 6                        list
# num_samples              562                      int
# num_nodes                4383                     int


In [2]:
#### convenience variables ####
_embedding_size = 16

# from mask (init)
_mask = brca.mask
_num_nodes, _num_sets = _mask.shape

# from batch (forward)
_batch_size = int(_batch.x.shape[0]/_num_nodes)
_num_node_features = _batch.x.shape[1] # or brca.num_node_features
_x = _batch.x.view(_batch.batch_size, int(_batch.x.shape[0]/_batch.batch_size), -1)

---

In [3]:
from modules.utils import reshape_x
from modules.model import cloneable, SequentialModel

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing

from torch import Tensor
from typing import Literal, Optional

In [4]:
@cloneable
class TanhGATConv(MessagePassing):
    def __init__(self, in_channels:int, out_channels:int, edge_channels:Optional[int]=None, temperature:float=1.0, eps=1e-8, *args, **kwargs):
        super().__init__(aggr='add')
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.temperature = temperature
        self.eps = eps

        self.lin_q = nn.Linear(in_channels, out_channels)
        self.lin_kv = nn.Linear(in_channels, 2*out_channels)
        self.norm = nn.LayerNorm(out_channels)

        if edge_channels is not None:
            self.lin_e = nn.Linear(edge_channels, 6*out_channels)
        else:
            self.lin_e = None

    def forward(self, x:Tensor, edge_index:Tensor, edge_attr:Optional[Tensor]=None, *args, **kwargs):
        # get qkv
        q = self.lin_q(x)
        k, v = self.lin_kv(x).chunk(2, dim=-1)

        # transformer-like attention module
        out = self.propagate(edge_index, q=q, k=k, v=v, edge_attr=edge_attr)

        # add & norm
        out = self.norm(out + q)

        return out

    def message(self, q_i, k_j, v_j, edge_attr, index, size_i,):
        # get e_qkv (if applicable)
        if edge_attr is not None:
            # compute modulation values from edges
            q_e, k_e, v_e, q_gate, k_gate, v_gate = self.lin_e(edge_attr).chunk(6, dim=-1)

            # edge gated node modulation
            q_i = q_i + q_e * torch.sigmoid(q_gate)
            k_j = k_j + k_e * torch.sigmoid(k_gate)
            v_j = v_j + v_e * torch.sigmoid(v_gate)

        # vaswani dot product attention
        att = (q_i * k_j).sum(dim=-1) / self.out_channels

        # tanh L1 norm attention
        att = torch.tanh(att/self.temperature)
        att_sum = torch.zeros(size_i, device=att.device).scatter_add(0, index, att.abs())
        att = att / (att_sum[index] + self.eps)

        # save attention for analyses
        self._prev_att = att.detach()

        # apply attention to message
        return att.unsqueeze(-1) * v_j


---

In [5]:
def attn_dims(embed_dim:Optional[int], head_dim:Optional[int], num_heads:int):
    # none specified; assert error
    assert (embed_dim is not None) or (head_dim is not None), 'one of [embed_dim, head_dim] must be specified'

    # both specified; lin_out reshapes head to embed
    if (embed_dim is not None) and (head_dim is not None):
        assert embed_dim // num_heads == head_dim, 'transformer dims incompatible, (embed_dim // num_heads == head_dim) must be true'
        return embed_dim, head_dim

    # embed_dim specified; head = embed / num_heads
    elif embed_dim is not None:
        assert embed_dim % num_heads == 0, 'embed_dim must be divisible by num_heads'
        head_dim = embed_dim // num_heads

    # head_dim specified; embed = head * num_heads
    elif head_dim is not None:
        embed_dim = head_dim * num_heads

    return embed_dim, head_dim

In [ ]:
@cloneable
class AttnSetPooling(nn.Module):
    def __init__(self, mask:Tensor, pool_method:Literal['add','mean']='mean', embed_dim:int=None, head_dim:int=None, num_heads:int=1, dropout:float=0.0, twin:bool=True, *args, **kwargs):
        '''
        mask in (nodes, sets)
        '''
        super().__init__(*args, **kwargs)
        self.embed_dim, _ = attn_dims(embed_dim, head_dim, num_heads)
        self.twin = twin

        # vars
        self.mask = mask.T # convert to (sets, nodes) for attn_mask in (q, kv)
        self.num_sets, self.num_nodes = self.mask.shape
        self.pool_method = pool_method
        self.set_counts = self.mask.sum(dim=-1, keepdim=True).clamp(min=1)  # sum across nodes, in (sets, 1); clamp to prevent div0
        
        # layers
        self.attn_layer = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, self.embed_dim),
            nn.ReLU(),
            nn.Linear(self.embed_dim, self.embed_dim)
        )
        self.norm2 = nn.LayerNorm(self.embed_dim)

    def attn(self, q, kv, need_weights, attn_mask=None):
        # get attention
        attn_out, weights = self.attn_layer(
            query=q,
            key=kv,
            value=kv,
            attn_mask = attn_mask,
            need_weights = need_weights
        )

        # get output
        h = self.norm1(attn_out + q) # add+norm (q is resid, before attn)
        ffn_out = self.ffn(h) # ffn
        h = self.norm2(ffn_out + h) # add+norm (h is resid, before ffn)

        return h, weights

    def forward(self, x:Tensor, need_weights:bool=False, as_dict:bool=True, *args, **kwargs):
        '''
        x in (b*n,f) or (b,n,f)
        attn_mask in (n, s) or (b * heads, n, s)
        '''
        # convert x to (b,n,F) if applicable
        x_node = reshape_x(x, 'b,n,f', num_nodes=self.num_nodes, num_node_features=self.embed_dim)

        # pool
        x_set = self.mask @ x_node # (b,s,n) @ (b,n,E) -> (b,s,E)
        if self.pool_method == 'mean':
            x_set = x_set / self.set_counts

        # get masked node-set attn
        self.attn_mask = (self.mask == 0)
        h_set, set_weights = self.attn(
            q = x_set,
            kv = x_node,
            attn_mask = self.attn_mask,
            need_weights = need_weights
        )

        # get node self-attn
        if self.twin:
            h_node, node_weights = self.attn(
                q = x_node,
                kv = x_node,
                need_weights = need_weights
            )
        else:
            h_node = None
            node_weights = None

        # format return
        if as_dict:
            return {'h_set':h_set, 'h_node':h_node, 'set_weights':set_weights, 'node_weights':node_weights}
        else:
            return h_set, h_node, set_weights, node_weights

In [ ]:
# gat = TanhGATConv(
#     in_channels=brca.num_node_features,
#     out_channels=_embedding_size,
#     edge_channels=brca.num_edge_features
# )

# asp = AttnSetPooling(
#     mask=brca.mask,
#     head_dim=_embedding_size
# )

# _node_emb = gat(_batch.x, _batch.edge_index, _batch.edge_attr)

# _set_emb = asp(_node_emb, twin=True)
# _set_emb['h_set'].shape

torch.Size([64, 305, 16])

---

In [31]:
@cloneable
class AttnGlobalPooling(nn.Module):
    def __init__(self, embed_dim:int=None, head_dim:int=None, num_heads:int=1, dropout:float=0.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        embed_dim, _ = attn_dims(embed_dim, head_dim, num_heads)
        self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim))

        # layers
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x:Tensor, need_weights:bool=False, as_dict:bool=True, *args, **kwargs):
        '''
        x in (b,n,E) or (b,s,E); returns z in (b,E) via cls-token attention
        '''
        batch_size = x.shape[0]

        # expand cls
        cls = self.cls_token.expand(batch_size, -1, -1)

        # get cross attn, in (b,1,E)
        attn_out, weights = self.attn(
            query = cls,
            key = x,
            value = x,
            need_weights = need_weights
        )

        # get output
        z = self.norm1(attn_out + cls) # add+norm (cls is resid, before attn)
        ffn_out = self.ffn(z) # ffn
        z = self.norm2(ffn_out + z) # add+norm (z is resid before ffn)

        # squeeze (b,1,E) -> (b,E)
        z = z.squeeze(1)

        # format return
        return {'z':z, 'weights':weights} if as_dict else (z, weights)

        

In [ ]:
tgat = TanhGATConv(
    in_channels=brca.num_node_features,
    out_channels=2*_embedding_size, # need to handle this in model
    edge_channels=brca.num_edge_features
)

asp = AttnSetPooling(
    mask=brca.mask,
    head_dim=_embedding_size,
    num_heads=2
)

agp = AttnGlobalPooling(
    head_dim=_embedding_size,
    num_heads=2
)

_node_emb = tgat(_batch.x, _batch.edge_index, _batch.edge_attr) # out (Tensor) = (b*n,F)
_set_emb = asp(_node_emb) # out (dict) = h_ in (b,n,F) or (b,s,F); _weights in (n,n) or (s,s) ??
_samp_emb = agp(_set_emb['h_node']) # out (dict) = z in (b, F)
_samp_emb['z'].shape
# # print(_samp_emb.shape)
# _samp_emb

torch.Size([64, 32])